# NB3 — Baselines (same split, same protocol)

Trains standard multimodal baselines on the **identical** Sen1Floods11 split and
training recipe as LGMF, so NB4 can compare like-for-like — especially the
**graceful-degradation gap**:

- `unet_naive` — concat U-Net trained on both modalities only (no dropout). Strong
  on 'both', collapses when a modality is missing (no gate/curriculum).
- `unet_drop`  — concat U-Net trained WITH the modality-dropout curriculum (a fair
  robustness baseline; still no gate/FiLM/consistency).
- `manet_drop` — concat MANet + curriculum (a heavier architecture point).

Specialized architectures (DMCF-Net, SMAGNet, CBC-SLP) are compared via their
published numbers / their own datasets — faithful reimplementation is out of
scope for a fair automated run.

**Attach:** `lgmf_c` (v5), Sen1Floods11 imagery, `lgmf-index`. Internet: On
(smp downloads encoder weights). Runtime ~3h (9 runs). Resume guard included.


In [ ]:
import sys, os, glob, json, random, time, warnings
warnings.filterwarnings("ignore")
init = glob.glob("/kaggle/input/**/lgmf/__init__.py", recursive=True)
assert init, "attach lgmf_c v5"
sys.path.insert(0, os.path.dirname(os.path.dirname(init[0])))
try:
    import segmentation_models_pytorch  # noqa
except ImportError:
    os.system("pip install -q segmentation-models-pytorch")
import numpy as np, torch
from torch.utils.data import DataLoader
import lgmf
from lgmf.config import LGMFConfig
from lgmf.dataset import Sen1Floods11
from lgmf.baselines import build_baseline
from lgmf.ema import ModelEMA
from lgmf.engine import train_one_epoch, evaluate
print("lgmf", lgmf.__version__)
assert lgmf.__version__ >= "5.0.0", "attach lgmf_c v5"

idx = os.path.dirname(glob.glob("/kaggle/input/**/norm_stats.json", recursive=True)[0])
stats = json.load(open(f"{idx}/norm_stats.json"))
print("index:", idx, "| split:", stats["split_source"])

## Config: baselines x seeds

In [ ]:
SEEDS  = [42, 1, 2]
EPOCHS = 45
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pin = device.type == "cuda"
regs = ["both", "s1_only", "s2_only"]

# (name, builder kwargs, cfg overrides). All share LGMF's recipe except the
# ablation-specific bits: baselines have NO consistency loss.
BASELINES = [
    ("unet_naive", dict(arch="unet",  encoder="mobilenet_v2"),
        dict(use_curriculum=False, fixed_regime_probs=(1.0, 0.0, 0.0))),
    ("unet_drop",  dict(arch="unet",  encoder="mobilenet_v2"), dict()),
    ("manet_drop", dict(arch="manet", encoder="mobilenet_v2"), dict()),
]

def make_cfg(name, seed, ov):
    c = LGMFConfig()
    c.epochs, c.lr, c.batch_size = EPOCHS, 3e-4, 8
    c.ema_decay, c.loss_type = 0.99, "focal_tversky"
    c.consistency_weight = 0.0            # baselines: no self-consistency
    c.num_workers = 0                     # avoid rasterio fork deadlock
    c.split_dir, c.ckpt_dir = idx, "/kaggle/working/ckpt_base"
    c.s1_mean, c.s1_std = stats["s1_mean"], stats["s1_std"]
    c.s2_mean, c.s2_std = stats["s2_mean"], stats["s2_std"]
    c.seed, c.run_name = seed, name
    for k, v in ov.items(): setattr(c, k, v)
    return c

## Trainer (same determinism + EMA as NB2b)

In [ ]:
def seed_all(sd):
    random.seed(sd); np.random.seed(sd); torch.manual_seed(sd); torch.cuda.manual_seed_all(sd)
    torch.backends.cudnn.benchmark = False

def run_one(cfg, build_kw):
    seed_all(cfg.seed)
    tr_ds = Sen1Floods11(cfg, f"{idx}/train_index.json", augment=True)
    va_ds = Sen1Floods11(cfg, f"{idx}/val_index.json",   augment=False)
    tr = DataLoader(tr_ds, batch_size=cfg.batch_size, shuffle=True, num_workers=0,
                    pin_memory=pin, drop_last=True)
    va = DataLoader(va_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=0, pin_memory=pin)
    model = build_baseline(pretrained=True, in_ch=8, **build_kw).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg.epochs)
    scaler = torch.amp.GradScaler("cuda") if (cfg.amp and device.type == "cuda") else None
    ema = ModelEMA(model, cfg.ema_decay) if cfg.use_ema else None
    em = ema.module if ema else model
    best, best_val = -1.0, None
    for epoch in range(cfg.epochs):
        train_one_epoch(model, tr, opt, cfg, epoch, device, scaler=scaler, ema=ema)
        sched.step()
        val = evaluate(em, va, cfg, device)
        sc = float(np.mean([val[r]["iou"] for r in regs]))
        if sc > best:
            best = sc
            best_val = {r: {"iou": val[r]["iou"], "f1": val[r]["f1"],
                            "flood_only_iou": val[r].get("flood_only_iou")} for r in regs}
            D = os.path.join(cfg.ckpt_dir, cfg.run_name, f"seed{cfg.seed}"); os.makedirs(D, exist_ok=True)
            torch.save({"model": model.state_dict(),
                        "ema": ema.module.state_dict() if ema else None,
                        "baseline": {"in_ch": 8, **build_kw},   # lets NB4 rebuild
                        "epoch": epoch, "val": best_val, "cfg": dict(cfg.__dict__)},
                       os.path.join(D, "best.pt"))
    return best, best_val

## Run all (baseline x seed)

In [ ]:
results = {name: [] for name, _, _ in BASELINES}
t0 = time.time()
for name, build_kw, ov in BASELINES:
    for sd in SEEDS:
        D = os.path.join("/kaggle/working/ckpt_base", name, f"seed{sd}")
        rj = os.path.join(D, "result.json")
        if os.path.exists(rj):
            rec = json.load(open(rj)); best, bv = rec["best"], rec["val"]
            print(f"[skip] {name:11s} seed{sd}  mean {best:.3f}")
        else:
            best, bv = run_one(make_cfg(name, sd, ov), build_kw)
            os.makedirs(D, exist_ok=True)
            json.dump({"best": best, "val": bv, "baseline": {"in_ch": 8, **build_kw}},
                      open(rj, "w"), indent=2, default=str)
            print(f"[done] {name:11s} seed{sd}  mean {best:.3f}   both {bv['both']['iou']:.3f} "
                  f"s1 {bv['s1_only']['iou']:.3f} s2 {bv['s2_only']['iou']:.3f}  "
                  f"({(time.time()-t0)/60:.0f} min)")
        results[name].append({"seed": sd, "best": best, "val": bv})
print(f"\nbaselines done in {(time.time()-t0)/60:.0f} min")

## Baseline table (val, 3 regimes, mean ± std)

In [ ]:
def ms(name, reg):
    a = np.array([r["val"][reg]["iou"] for r in results[name]])
    return a.mean(), a.std()
def cell(name, reg): m, s = ms(name, reg); return f"{m:.3f}±{s:.3f}"

print("=" * 78)
print(f"{'baseline':12s} {'both':>13s} {'s1_only':>13s} {'s2_only':>13s}   {'drop both->s1':>14s}")
print("-" * 78)
for name, _, _ in BASELINES:
    drop = ms(name, "both")[0] - ms(name, "s1_only")[0]     # graceful-degradation gap
    print(f"{name:12s} {cell(name,'both'):>13s} {cell(name,'s1_only'):>13s} "
          f"{cell(name,'s2_only'):>13s}   {drop:>14.3f}")
print("=" * 78)
print("'drop both->s1' = how far SAR-only falls below both-modality. LGMF's job is a SMALL drop;")
print("watch unet_naive fall hardest (trained on 'both' only). NB4 plots this vs LGMF.")

json.dump({"seeds": SEEDS, "epochs": EPOCHS, "results": results},
          open("/kaggle/working/baseline_summary.json", "w"), indent=2, default=str)
print("saved baseline_summary.json")

## Next
1. Save `/kaggle/working/ckpt_base` as a dataset `lgmf-baselines`.
2. **NB4** loads LGMF (`lgmf-ckpts`) + baselines (`lgmf-baselines`), evaluates all on
   the **test** split, and produces: the main results table, the efficiency Pareto
   plot, and the degradation curve (both->s1->s2) — the headline graceful-degradation figure.
